In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline

In [3]:
# Charger les données
df = pd.read_csv(r"C:\Users\zizou\OneDrive\Desktop\stage 3ème\day 2\csvfiles\dataframefinale.csv",sep=";")
# Afficher les premières lignes
df.head()

# Vérifier les valeurs manquantes
df.isnull().sum()

# Supprimer les valeurs manquantes si nécessaire
df = df.dropna()

# Sélection des caractéristiques et cible
X = df.drop(columns=['Montant', 'dataloadingdate', 'JourSemaine'])  # Exemple de features
y = df['Montant']

# Conversion des catégories si nécessaire (one-hot encoding)
X = pd.get_dummies(X, drop_first=True)

# Séparation train-test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'SVR': SVR()
}

In [5]:
results = {}

for name, model in models.items():
    # Entraînement
    model.fit(X_train, y_train)
    
    # Prédiction
    y_pred = model.predict(X_test)
    
    # Calcul RMSE
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    # Stockage résultat
    results[name] = {'RMSE': rmse}
    
    print(f"{name}: RMSE = {rmse:.4f}")

Linear Regression: RMSE = 5.9710
Ridge Regression: RMSE = 5.9629
Lasso Regression: RMSE = 7.0207
Decision Tree: RMSE = 3.2213
Random Forest: RMSE = 2.1374
Gradient Boosting: RMSE = 4.0477
SVR: RMSE = 7.0799


In [6]:
# Normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

for name, model in models.items():
    # Entraînement avec données normalisées
    model.fit(X_train_scaled, y_train)
    
    # Prédiction
    y_pred = model.predict(X_test_scaled)
    
    # Calcul RMSE
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    # Stockage résultat
    results[name]['RMSE_scaled'] = rmse
    
    print(f"{name} (scaled): RMSE = {rmse:.4f}")

Linear Regression (scaled): RMSE = 941991785203.0297
Ridge Regression (scaled): RMSE = 5.9728
Lasso Regression (scaled): RMSE = 7.0568
Decision Tree (scaled): RMSE = 3.2220
Random Forest (scaled): RMSE = 2.1276
Gradient Boosting (scaled): RMSE = 4.0476
SVR (scaled): RMSE = 5.8447


In [7]:
#la selection des 3 meilleurs features 
selector = SelectKBest(f_regression, k=3)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

for name, model in models.items():
    # Entraînement avec features sélectionnées
    model.fit(X_train_selected, y_train)
    
    # Prédiction
    y_pred = model.predict(X_test_selected)
    
    # Calcul RMSE
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    # Stockage résultat
    results[name]['RMSE_selected'] = rmse
    
    print(f"{name} (selected features): RMSE = {rmse:.4f}")

Linear Regression (selected features): RMSE = 6.9094
Ridge Regression (selected features): RMSE = 6.9094
Lasso Regression (selected features): RMSE = 7.0568
Decision Tree (selected features): RMSE = 6.5846
Random Forest (selected features): RMSE = 5.8951
Gradient Boosting (selected features): RMSE = 5.5748
SVR (selected features): RMSE = 6.8227


In [8]:
# Paramètres pour chaque modèle
param_grids = {
    'Ridge Regression': {'alpha': [0.1, 1, 10, 100]},
    'Lasso Regression': {'alpha': [0.1, 1, 10, 100]},
    'Decision Tree': {'max_depth': [None, 5, 10, 20]},
    'Random Forest': {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20]},
    'Gradient Boosting': {'n_estimators': [50, 100], 'learning_rate': [0.01, 0.1, 1]},
    'SVR': {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}
}

for name in param_grids.keys():
    # Création du pipeline
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(f_regression, k=10)),
        ('model', models[name])
    ])
    
    # Recherche des meilleurs paramètres
    grid_search = GridSearchCV(pipeline, 
                             {'model__' + key: value for key, value in param_grids[name].items()},
                             cv=5, 
                             scoring='neg_mean_squared_error')
    
    grid_search.fit(X_train, y_train)
    # Meilleur modèle
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    # Calcul RMSE
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    # Stockage résultat
    results[name]['RMSE_tuned'] = rmse
    
    print(f"{name} (tuned): RMSE = {rmse:.4f}")
    print(f"Meilleurs paramètres: {grid_search.best_params_}")

Ridge Regression (tuned): RMSE = 6.7998
Meilleurs paramètres: {'model__alpha': 0.1}
Lasso Regression (tuned): RMSE = 6.8050
Meilleurs paramètres: {'model__alpha': 0.1}
Decision Tree (tuned): RMSE = 5.6042
Meilleurs paramètres: {'model__max_depth': 10}
Random Forest (tuned): RMSE = 5.4517
Meilleurs paramètres: {'model__max_depth': 10, 'model__n_estimators': 50}
Gradient Boosting (tuned): RMSE = 5.4176
Meilleurs paramètres: {'model__learning_rate': 0.1, 'model__n_estimators': 100}
SVR (tuned): RMSE = 6.3331
Meilleurs paramètres: {'model__C': 10, 'model__kernel': 'rbf'}


In [9]:
results_df = pd.DataFrame(results).T
results_df

,RMSE,RMSE_scaled,RMSE_selected,RMSE_tuned
Linear Regression,5.970980,9.419918e+11,6.909431,NaN
Ridge Regression,5.962883,5.972801e+00,6.909435,6.799771
Lasso Regression,7.020683,7.056764e+00,7.056764,6.805027
Decision Tree,3.221271,3.221986e+00,6.584640,5.604205
Random Forest,2.137407,2.127571e+00,5.895091,5.451655
Gradient Boosting,4.047670,4.047637e+00,5.574837,5.417581
SVR,7.079863,5.844704e+00,6.822746,6.333072


In [10]:
# Trouver le meilleur modèle (le plus petit RMSE après tuning)
best_model_name = results_df['RMSE_tuned'].idxmin()
best_rmse = results_df.loc[best_model_name, 'RMSE_tuned']

print(f"Meilleur modèle: {best_model_name} avec RMSE = {best_rmse:.4f}")

Meilleur modèle: Gradient Boosting avec RMSE = 5.4176
